<a href="https://colab.research.google.com/github/AhmedGDeeb/DataTalksClub/blob/main/machine-learning-zoomcamp/cohorts/2026/homework/03-classification/homework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, mutual_info_score

# Utils

# Get data

In [ ]:
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"

df = pd.read_csv(url)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               4852 non-null   object 
 1   industry                  4760 non-null   object 
 2   employment_status         4806 non-null   object 
 3   location                  4792 non-null   object 
 4   annual_income             4631 non-null   float64
 5   number_of_courses_viewed  5000 non-null   int64  
 6   interaction_count         5000 non-null   int64  
 7   lead_score                4965 non-null   float64
 8   converted                 5000 non-null   int64  
dtypes: float64(2), int64(3), object(4)
memory usage: 351.7+ KB


### Data preparation
- Check if the missing values are presented in the features.

- If there are missing values:

   - For categorical features, replace them with 'NA'

   - For numerical features, replace with with 0.0

In [9]:
df.isnull().sum()

,0
lead_source,148
industry,240
employment_status,194
location,208
annual_income,369
number_of_courses_viewed,0
interaction_count,0
lead_score,35
converted,0


In [10]:
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
numerical_cols = df.select_dtypes(exclude=['object']).columns.tolist()

# Remove target from numerical list if it's there (it's int/bool, not object)
if 'converted' in numerical_cols:
    numerical_cols.remove('converted')

print("Categorical:", categorical_cols)
print("Numerical:", numerical_cols)

df[categorical_cols] = df[categorical_cols].fillna('NA')
df[numerical_cols] = df[numerical_cols].fillna(0.0)

Categorical: ['lead_source', 'industry', 'employment_status', 'location']
Numerical: ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']


In [11]:
df.isnull().sum()

,0
lead_source,0
industry,0
employment_status,0
location,0
annual_income,0
number_of_courses_viewed,0
interaction_count,0
lead_score,0
converted,0


## Split Data

plit the data
Split the data with these exact calls:

```python
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)
```

Make sure that the target value `converted` is not in your dataframe.

In [21]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

del df_train['converted']
del df_val['converted']
del df_test['converted']

print("Train:", df_train.shape, "Val:", df_val.shape, "Test:", df_test.shape)

Train: (3000, 8) Val: (1000, 8) Test: (1000, 8)


# Question 1

What is the most frequent observation (mode) for the column industry?

- NA

- technology

- healthcare

- retail

In [12]:
mode_industry = df['industry'].mode()[0]

# Answer

In [13]:
print(mode_industry)

technology


# Question 2

Create the correlation matrix for the numerical features of your dataset.
In a correlation matrix, you compute the correlation coefficient between every pair of features.

What are the two features that have the biggest correlation?

- `interaction_count` and `lead_score`

- `number_of_courses_viewed` and `lead_score`

- `number_of_courses_viewed` and `interaction_count`

- `annual_income` and `interaction_count`

Only consider the pairs above when answering this question.

In [15]:
# Check only the pairs listed in the question
pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count'),
]

corr_matrix = df[numerical_cols].corr()

# Store correlations in a dict
corr_dict = {}
for a, b in pairs:
    if a in corr_matrix.columns and b in corr_matrix.columns:
        corr_dict[(a, b)] = corr_matrix.loc[a, b]
        print(f"corr({a}, {b}) = {corr_matrix.loc[a, b]:.4f}")

corr(interaction_count, lead_score) = 0.9157
corr(number_of_courses_viewed, lead_score) = 0.7572
corr(number_of_courses_viewed, interaction_count) = 0.7216
corr(annual_income, interaction_count) = 0.1228


# Answer

In [19]:
max_pair = max(corr_dict, key=corr_dict.get)

" and ".join(list(max_pair))

'interaction_count and lead_score'

# Question 3

- Calculate the mutual information score between converted and other categorical variables in the dataset. Use the training set only.

- Round the scores to 2 decimals using round(score, 2).

Which of these variables has the biggest mutual information score?

- industry

- location

- lead_source

- employment_status

In [22]:
def mutual_info_converted_score(series):
    return mutual_info_score(series, y_train)

cat_cols_for_mi = ['industry', 'location', 'lead_source', 'employment_status']

mi_scores = {}
for col in cat_cols_for_mi:
    mi = mutual_info_converted_score(df_train[col])
    mi_scores[col] = round(mi, 2)

print("Q3 - Mutual information scores:")
for k, v in mi_scores.items():
    print(f"  {k}: {v}")

Q3 - Mutual information scores:
  industry: 0.0
  location: 0.0
  lead_source: 0.03
  employment_status: 0.02
Biggest MI: lead_source 0.03


# Answer

In [23]:
best_mi = max(mi_scores, key=mi_scores.get)
best_mi, mi_scores[best_mi]

best_mi

'lead_source'

# Question 4

- Now let's train a logistic regression.

- Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.

- Fit the model on the training dataset.

  - To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
  ```python
  model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
  ```

- Calculate the accuracy on the validation dataset and round it to 2 decimal digits.

What accuracy did you get?

- 0.55

- 0.65

- 0.75

- 0.85

In [25]:
def prepare_features(df):
    """Convert dataframe rows to list of dicts for DictVectorizer."""
    return df.to_dict(orient='records')

train_dicts = prepare_features(df_train)
val_dicts = prepare_features(df_val)

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)

# Answer

In [26]:
acc_q4 = round(accuracy_score(y_val, y_pred), 2)

acc_q4

0.65

# Question 5

- Let's find the least useful feature using the feature elimination technique.

- Train a model using the same features and parameters as in Q4 (without rounding).

- Now exclude each feature from this set and train a model without it. Record the accuracy for each model.

- For each feature, calculate the difference between the original accuracy and the accuracy without the feature.

Which of following feature has the smallest difference?

- 'lead_source'

- 'number_of_courses_viewed'

- 'interaction_count'

> Note: The difference doesn't have to be positive.

In [27]:
base_acc = accuracy_score(y_val, y_pred)  # unrounded

features_to_test = ['lead_source', 'number_of_courses_viewed', 'interaction_count']

diff_results = {}
for feat in features_to_test:
    # Drop the feature from train/val dataframes
    df_tr = df_train.drop(columns=[feat])
    df_vl = df_val.drop(columns=[feat])

    tr_dicts = prepare_features(df_tr)
    vl_dicts = prepare_features(df_vl)

    dv_f = DictVectorizer(sparse=True)
    X_tr = dv_f.fit_transform(tr_dicts)
    X_vl = dv_f.transform(vl_dicts)

    m = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    m.fit(X_tr, y_train)
    acc = accuracy_score(y_val, m.predict(X_vl))

    diff = base_acc - acc
    diff_results[feat] = diff
    print(f"{feat}: acc={acc:.6f}, diff={diff:.6f}")

lead_source: acc=0.642000, diff=0.003000
number_of_courses_viewed: acc=0.643000, diff=0.002000
interaction_count: acc=0.601000, diff=0.044000
Q5 - Smallest difference feature: number_of_courses_viewed 0.0020000000000000018


# Answer

In [28]:
smallest = min(diff_results, key=lambda k: abs(diff_results[k]))

smallest

'number_of_courses_viewed'

# Question 6

- Now let's train a regularized logistic regression.

- Let's try the following values of the parameter `C`: `[0.000001, 0.00001, 0.0001, 0.001]`.

Train models using all the features as in Q4.

Calculate the accuracy on the validation dataset and round it to 3 decimal digits.

Which of these `C` leads to the best accuracy on the validation set?

- 0.000001

- 0.00001

- 0.0001

- 0.001

> Note: If there are multiple options, select the smallest `C`.

In [29]:
C_values = [0.000001, 0.00001, 0.0001, 0.001]

results = {}
for C in C_values:
    m = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    m.fit(X_train, y_train)
    acc = round(accuracy_score(y_val, m.predict(X_val)), 3)
    results[C] = acc
    print(f"C={C}: accuracy={acc}")

C=1e-06: accuracy=0.598
C=1e-05: accuracy=0.598
C=0.0001: accuracy=0.613
C=0.001: accuracy=0.645


# Answer

In [30]:
# Pick best accuracy; if tie, pick smallest C
best_C = None
best_acc = -1
for C in sorted(results.keys()):
    if results[C] > best_acc:
        best_acc = results[C]
        best_C = C

best_C

0.001

# Submit the results


Submit your results here: https://courses.datatalks.club/ml-zoomcamp-2026/homework/hw03

The numerical options are calculated from the pinned 2026 release. Use the value that matches your calculation; do not choose a merely close value.

